In [7]:
import os
import re
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr
import plotly.io as pio
from typing import Dict, List, Optional

pio.renderers.default = "plotly_mimetype"

# --- Helper Functions (assuming they are defined in a previous cell) ---
def parse_config(file_path: str) -> Dict[str, str]:
    """Parses a 'key: value' or 'key = value' configuration file."""
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = re.split(r'[:=]', line, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except Exception:
        # Silently ignore if file not found or parsing fails
        pass
    return params

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except (ValueError, IndexError):
        return 0.0

# --- Main Data Processing and Plotting Cell ---

BASE_OUTPUT_DIR = '/app/astra-sim/upc/output/comparison_run'
# TOPOLOGIES = ['Jellyfish', 'FoldedClos', 'Dragonfly']
TOPOLOGIES = ['FoldedClos128']
WORKLOAD_GROUP = 'T5_Small_grouped_ecmp'
WORKLOAD_GROUP = 'T5_Small_grouped_128' # The parent folder for the workloads

all_results = []

for topo in TOPOLOGIES:
    topo_path = os.path.join(BASE_OUTPUT_DIR, topo, WORKLOAD_GROUP)
    if not os.path.isdir(topo_path):
        print(f"Directory not found for topology {topo}, skipping.")
        continue

    # A workload is a specific model parallelization strategy, e.g., "T5_Small_multiple_1_8_2_1_0..."
    for workload_name in os.listdir(topo_path):
        workload_path = os.path.join(topo_path, workload_name)
        if not os.path.isdir(workload_path):
            continue

        # This dictionary will hold the consolidated results for one workload
        workload_results = {'workload': workload_name, 'topology': topo}
        
        # Each workload folder contains multiple run directories, one for each simulator
        run_dirs = [d for d in os.listdir(workload_path) if d.startswith('run_')]

        for run_dir_name in run_dirs:
            run_path = os.path.join(workload_path, run_dir_name)

            # Determine which simulator this run directory is for
            sim_type = None
            if os.path.isdir(os.path.join(run_path, 'g2')):
                sim_type = 'g2'
            elif os.path.isdir(os.path.join(run_path, 'ns3')):
                sim_type = 'ns3'
            elif os.path.isdir(os.path.join(run_path, 'analytical_unaware')):
                sim_type = 'analytical_unaware'
            
            if not sim_type:
                continue # Skip if this run directory doesn't contain a known sim output

            sim_path = os.path.join(run_path, sim_type)
            sim_name = {'g2': 'G2', 'ns3': 'NS3', 'analytical_unaware': 'Analytical'}[sim_type]

            # 1. Extract Estimated Execution Time (from timing.csv)
            timing_file = next((os.path.join(sim_path, f) for f in os.listdir(sim_path) if 'trace_matched_timing.csv' in f), None)
            max_time_ns = None
            if timing_file:
                try:
                    df_timing = pd.read_csv(timing_file)
                    if 'callback_tick' in df_timing.columns:
                        max_time_ns = df_timing['callback_tick'].max()
                except Exception as e:
                    print(f"Error reading {timing_file}: {e}")
            
            # 2. Extract Simulation Time (from run_summary.txt)
            summary_params = parse_config(os.path.join(run_path, 'run_summary.txt'))
            sim_time_sec = parse_runtime(summary_params.get('total runtime', '0'))

            # Store results in the workload's dictionary
            workload_results[f'{sim_name}_Est_Exec_Time_ns'] = max_time_ns
            workload_results[f'{sim_name}_Sim_Time_sec'] = sim_time_sec
        
        # Only add the workload if it has data
        if len(workload_results) > 2:
             all_results.append(workload_results)

# --- Create DataFrame, Plots, and Summary Tables ---
if all_results:
    df = pd.DataFrame(all_results)
    # Ensure all required simulator data is present for a workload before processing
    required_cols = ['G2_Est_Exec_Time_ns', 'NS3_Est_Exec_Time_ns', 'Analytical_Est_Exec_Time_ns', 'G2_Sim_Time_sec', 'NS3_Sim_Time_sec', 'Analytical_Sim_Time_sec']
    df.dropna(subset=required_cols, inplace=True)
    df.sort_values(by=['topology', 'workload'], inplace=True)
    
    # Create output directory for plots
    plot_output_dir = os.path.join(BASE_OUTPUT_DIR, "summary_plots")
    os.makedirs(plot_output_dir, exist_ok=True)

    # Precompute error and speedup columns
    df['G2 Error (%)'] = ((df['G2_Est_Exec_Time_ns'] - df['NS3_Est_Exec_Time_ns']) / df['NS3_Est_Exec_Time_ns']) * 100
    df['AU Error (%)'] = ((df['Analytical_Est_Exec_Time_ns'] - df['NS3_Est_Exec_Time_ns']) / df['NS3_Est_Exec_Time_ns']) * 100
    df['G2 Speedup (x)'] = df['NS3_Sim_Time_sec'] / df['G2_Sim_Time_sec']
    df['AU Speedup (x)'] = df['NS3_Sim_Time_sec'] / df['Analytical_Sim_Time_sec']

    # --- Header for LaTeX tables ---
    print(f"\n\n{'='*80}\n--- Generated LaTeX Tables ---\n{'='*80}")

    for topo in df['topology'].unique():
        topo_df = df[df['topology'] == topo].copy()
        if topo_df.empty:
            continue
        
        # Sort by NS3 execution time as the ground truth
        topo_df = topo_df.sort_values(by='NS3_Est_Exec_Time_ns').reset_index(drop=True)
        
        # Shorten workload names for cleaner plots
        topo_df['short_workload'] = topo_df['workload'].str.replace('T5_Small_multiple_', '', regex=False)\
            .str.replace('_', '-', regex=False)\
            .str.replace('.seq-2048.batch-1024', '', regex=False)

        print(f"\n{'='*80}\n--- Results for Topology: {topo} ---\n{'='*80}")

        # --- Plot 1: Absolute Estimated Execution Time Comparison ---
        fig_abs = go.Figure()
        fig_abs.add_trace(go.Bar(x=topo_df['short_workload'], y=topo_df['NS3_Est_Exec_Time_ns'], name='NS3', marker_color='blue'))
        fig_abs.add_trace(go.Bar(x=topo_df['short_workload'], y=topo_df['G2_Est_Exec_Time_ns'], name='G2', marker_color='red'))
        fig_abs.add_trace(go.Bar(x=topo_df['short_workload'], y=topo_df['Analytical_Est_Exec_Time_ns'], name='Analytical', marker_color='green'))
        
        fig_abs.update_layout(
            title=f'Absolute Estimated Execution Time for {topo}',
            xaxis_title='Workload (Parallelization Strategy)',
            yaxis_title='Estimated Execution Time (ns)',
            barmode='group',
            xaxis_tickangle=-45,
            template='plotly_white',
            font=dict(size=16)
        )
        fig_abs.write_image(os.path.join(plot_output_dir, f"{topo}_absolute_time_comparison.pdf"), width=1400, height=600)
        fig_abs.show()

        # --- Plot 2: Ordering Comparison (Bump Chart) ---
        fig_bump = go.Figure()
        topo_df['NS3_rank'] = topo_df['NS3_Est_Exec_Time_ns'].rank(method='dense')
        topo_df['G2_rank'] = topo_df['G2_Est_Exec_Time_ns'].rank(method='dense')
        topo_df['AU_rank'] = topo_df['Analytical_Est_Exec_Time_ns'].rank(method='dense')

        for i, row in topo_df.iterrows():
            fig_bump.add_trace(go.Scatter(
                x=['G2', 'NS3', 'Analytical'],
                y=[row['G2_rank'], row['NS3_rank'], row['AU_rank']],
                mode='lines+markers',
                name=row['short_workload'],
                line=dict(color='black', width=1),
                showlegend=False
            ))
        
        fig_bump.update_layout(
            title=f'Performance Ranking Comparison for {topo}',
            xaxis_title='Simulator',
            yaxis_title='Rank (1 is fastest)',
            yaxis=dict(autorange='reversed', tick0=1, dtick=1), # Rank 1 at top
            template='plotly_white',
            font=dict(size=16)
        )
        fig_bump.write_image(os.path.join(plot_output_dir, f"{topo}_ranking_comparison.pdf"), width=800, height=600)
        fig_bump.show()

        # --- Plot 3: Normalized Estimated Execution Time Comparison ---
        fig_norm = go.Figure()
        fig_norm.add_trace(go.Bar(x=topo_df['short_workload'], y=[100] * len(topo_df), name='NS3', marker_color='blue'))
        fig_norm.add_trace(go.Bar(x=topo_df['short_workload'], y=(topo_df['G2_Est_Exec_Time_ns'] / topo_df['NS3_Est_Exec_Time_ns']) * 100, name='G2', marker_color='red'))
        fig_norm.add_trace(go.Bar(x=topo_df['short_workload'], y=(topo_df['Analytical_Est_Exec_Time_ns'] / topo_df['NS3_Est_Exec_Time_ns']) * 100, name='Analytical', marker_color='green'))
        fig_norm.update_layout(
            title=f'Normalized Estimated Time for {topo} (vs NS3)',
            xaxis_title='Workload (Parallelization Strategy)',
            yaxis_title='Relative Execution Time to NS3 (%)',
            barmode='group',
            xaxis_tickangle=-45,
            template='plotly_white',
            font=dict(size=16)
        )
        fig_norm.write_image(os.path.join(plot_output_dir, f"{topo}_normalized_time_comparison.pdf"), width=1400, height=600)
        fig_norm.show()

        # --- Spearman's Rank Correlation ---
        g2_corr, _ = spearmanr(topo_df['NS3_Est_Exec_Time_ns'], topo_df['G2_Est_Exec_Time_ns'])
        au_corr, _ = spearmanr(topo_df['NS3_Est_Exec_Time_ns'], topo_df['Analytical_Est_Exec_Time_ns'])
        print(f"\n--- Spearman's Rank Correlation for {topo} (vs NS3 Estimated Time) ---")
        print(f"A value close to 1.0 indicates that the simulator preserves the performance ranking of workloads.")
        print(f" - G2 vs NS3: {g2_corr:.4f}")
        print(f" - Analytical vs NS3: {au_corr:.4f}")

        # --- Summary Table Generation (for display in notebook) ---
        summary_df = pd.DataFrame()
        summary_df['Workload'] = topo_df['short_workload']
        summary_df['NS3 Est. Time (ns)'] = topo_df['NS3_Est_Exec_Time_ns']
        summary_df['G2 Est. Time (ns)'] = topo_df['G2_Est_Exec_Time_ns']
        summary_df['G2 Error (%)'] = topo_df['G2 Error (%)']
        summary_df['AU Est. Time (ns)'] = topo_df['Analytical_Est_Exec_Time_ns']
        summary_df['AU Error (%)'] = topo_df['AU Error (%)']
        summary_df['NS3 Sim. Time (s)'] = topo_df['NS3_Sim_Time_sec']
        summary_df['G2 Sim. Time (s)'] = topo_df['G2_Sim_Time_sec']
        summary_df['G2 Speedup (x)'] = topo_df['G2 Speedup (x)']
        summary_df['AU Sim. Time (s)'] = topo_df['Analytical_Sim_Time_sec']
        summary_df['AU Speedup (x)'] = topo_df['AU Speedup (x)']
        print(f"\n--- Summary Table for {topo} ---")
        with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
            display(summary_df.style.format({
                'NS3 Est. Time (ns)': '{:,.0f}', 'G2 Est. Time (ns)': '{:,.0f}', 'G2 Error (%)': '{:+.2f}%',
                'AU Est. Time (ns)': '{:,.0f}', 'AU Error (%)': '{:+.2f}%', 'NS3 Sim. Time (s)': '{:.2f}s',
                'G2 Sim. Time (s)': '{:.2f}s', 'G2 Speedup (x)': '{:.2f}x', 'AU Sim. Time (s)': '{:.2f}s',
                'AU Speedup (x)': '{:.2f}x'
            }))

        # --- LaTeX Table Generation (for papers) ---
        
        # Table 1: Estimated Execution Time
        est_df = pd.DataFrame()
        est_df['Workload'] = topo_df['short_workload']
        est_df['NS3 (s)'] = topo_df['NS3_Est_Exec_Time_ns'] / 1e9
        est_df['G2 (s)'] = topo_df['G2_Est_Exec_Time_ns'] / 1e9
        est_df['G2 Err. (%)'] = topo_df['G2 Error (%)']
        est_df['Analytic (s)'] = topo_df['Analytical_Est_Exec_Time_ns'] / 1e9
        est_df['Analytic Err. (%)'] = topo_df['AU Error (%)']
        
        # Add Average/MAPE row
        avg_row = pd.DataFrame([{
            'Workload': '\\textbf{Average / MAPE}',
            'NS3 (s)': est_df['NS3 (s)'].mean(),
            'G2 (s)': est_df['G2 (s)'].mean(),
            'G2 Err. (%)': est_df['G2 Err. (%)'].abs().mean(),
            'Analytic (s)': est_df['Analytic (s)'].mean(),
            'Analytic Err. (%)': est_df['Analytic Err. (%)'].abs().mean()
        }])
        est_df = pd.concat([est_df, avg_row], ignore_index=True)

        est_latex = est_df.to_latex(
            index=False,
            formatters={
                'NS3 (s)': "{:.3f}".format,
                'G2 (s)': "{:.3f}".format,
                'G2 Err. (%)': "{:+.2f}\%".format,
                'Analytic (s)': "{:.3f}".format,
                'Analytic Err. (%)': "{:+.2f}\%".format,
            },
            caption=f'Estimated Execution Times for {topo} Topology.',
            label=f'tab:est_times_{topo.lower()}',
            position='!htbp',
            column_format='lccccc', # l for left-aligned text, c for centered
            escape=False # To render \% correctly
        )
        # Add hline before the last row
        lines = est_latex.splitlines()
        lines.insert(-2, '\\hline')
        est_latex = '\n'.join(lines)
        est_latex = est_latex.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')


        print(f"\n--- LaTeX: Estimated Execution Time for {topo} ---")
        print(est_latex)

        # Table 2: Simulation Time
        sim_df = pd.DataFrame()
        sim_df['Workload'] = topo_df['short_workload']
        sim_df['NS3 (s)'] = topo_df['NS3_Sim_Time_sec']
        sim_df['G2 (s)'] = topo_df['G2_Sim_Time_sec']
        sim_df['G2 Speedup (x)'] = topo_df['G2 Speedup (x)']
        sim_df['Analytic (s)'] = topo_df['Analytical_Sim_Time_sec']
        sim_df['Analytic Speedup (x)'] = topo_df['AU Speedup (x)']

        # Add Average row
        avg_sim_row = pd.DataFrame([{
            'Workload': '\\textbf{Average}',
            'NS3 (s)': sim_df['NS3 (s)'].mean(),
            'G2 (s)': sim_df['G2 (s)'].mean(),
            'G2 Speedup (x)': sim_df['G2 Speedup (x)'].mean(),
            'Analytic (s)': sim_df['Analytic (s)'].mean(),
            'Analytic Speedup (x)': sim_df['Analytic Speedup (x)'].mean()
        }])
        sim_df = pd.concat([sim_df, avg_sim_row], ignore_index=True)

        sim_latex = sim_df.to_latex(
            index=False,
            formatters={
                'NS3 (s)': "{:.3f}".format,
                'G2 (s)': "{:.3f}".format,
                'G2 Speedup (x)': "{:.2f}x".format,
                'Analytic (s)': "{:.3f}".format,
                'Analytic Speedup (x)': "{:.2f}x".format,
            },
            caption=f'Simulation Times and Speedup for {topo} Topology.',
            label=f'tab:sim_times_{topo.lower()}',
            position='!htbp',
            column_format='lccccc', # l for left-aligned text, c for centered
            escape=False # To render x for speedup correctly
        )
        # Add hline before the last row
        lines = sim_latex.splitlines()
        lines.insert(-2, '\\hline')
        sim_latex = '\n'.join(lines)
        sim_latex = sim_latex.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')
        
        print(f"\n--- LaTeX: Simulation Time for {topo} ---")
        print(sim_latex)


    # --- Overall Average Absolute Error Calculation ---
    avg_g2_abs_error = df['G2 Error (%)'].abs().mean()
    avg_au_abs_error = df['AU Error (%)'].abs().mean()
    print(f"\n{'='*80}\n--- Overall Average Absolute Error (across all topologies) ---\n{'='*80}")
    print(f"Average G2 Absolute Error vs NS3: {avg_g2_abs_error:.2f}%")
    print(f"Average Analytical Absolute Error vs NS3: {avg_au_abs_error:.2f}%")

else:
    print("No results were found to process.")




--- Generated LaTeX Tables ---

--- Results for Topology: FoldedClos128 ---



--- Spearman's Rank Correlation for FoldedClos128 (vs NS3 Estimated Time) ---
A value close to 1.0 indicates that the simulator preserves the performance ranking of workloads.
 - G2 vs NS3: 0.9662
 - Analytical vs NS3: 0.9740

--- Summary Table for FoldedClos128 ---


,Workload,NS3 Est. Time (ns),G2 Est. Time (ns),G2 Error (%),AU Est. Time (ns),AU Error (%),NS3 Sim. Time (s),G2 Sim. Time (s),G2 Speedup (x),AU Sim. Time (s),AU Speedup (x)
0,16-2-4-1-1,"294,708,973","290,748,115",-1.34%,"73,408,884",-75.09%,29817.51s,31.10s,958.90x,4.19s,7121.62x
1,16-2-4-1-0,"341,492,273","284,520,631",-16.68%,"74,251,979",-78.26%,31401.58s,40.17s,781.81x,4.71s,6660.01x
2,16-4-2-1-0,"344,865,759","329,866,132",-4.35%,"88,479,050",-74.34%,37140.24s,39.23s,946.62x,3.97s,9359.09x
3,16-4-2-1-1,"345,586,378","355,266,931",+2.80%,"88,035,938",-74.53%,38149.17s,34.42s,1108.27x,4.23s,9018.19x
4,8-4-4-1-0,"385,385,444","386,111,040",+0.19%,"101,449,957",-73.68%,41628.30s,29.39s,1416.62x,4.04s,10306.28x
5,8-1-16-1-0,"397,901,892","574,718,172",+44.44%,"120,130,886",-69.81%,47609.63s,42.27s,1126.30x,3.94s,12072.60x
6,8-1-16-1-1,"418,183,271","464,785,480",+11.14%,"119,828,001",-71.35%,46833.71s,48.33s,969.03x,4.33s,10824.66x
7,16-8-1-1-0,"467,573,714","587,172,655",+25.58%,"127,720,590",-72.68%,58857.22s,55.30s,1064.32x,3.76s,15636.77x
8,4-2-16-1-1,"472,079,614","561,551,511",+18.95%,"133,908,839",-71.63%,52319.61s,39.38s,1328.48x,4.17s,12541.14x
9,4-4-8-1-0,"477,943,910","523,837,825",+9.60%,"121,571,843",-74.56%,48609.44s,34.89s,1393.05x,4.05s,11999.86x



--- LaTeX: Estimated Execution Time for FoldedClos128 ---
\begin{table}[!htbp]
\caption{Estimated Execution Times for FoldedClos128 Topology.}
\label{tab:est_times_foldedclos128}
\begin{tabular}{lccccc}
\hline
Workload & NS3 (s) & G2 (s) & G2 Err. (%) & Analytic (s) & Analytic Err. (%) \\
\hline
16-2-4-1-1 & 0.295 & 0.291 & -1.34\% & 0.073 & -75.09\% \\
16-2-4-1-0 & 0.341 & 0.285 & -16.68\% & 0.074 & -78.26\% \\
16-4-2-1-0 & 0.345 & 0.330 & -4.35\% & 0.088 & -74.34\% \\
16-4-2-1-1 & 0.346 & 0.355 & +2.80\% & 0.088 & -74.53\% \\
8-4-4-1-0 & 0.385 & 0.386 & +0.19\% & 0.101 & -73.68\% \\
8-1-16-1-0 & 0.398 & 0.575 & +44.44\% & 0.120 & -69.81\% \\
8-1-16-1-1 & 0.418 & 0.465 & +11.14\% & 0.120 & -71.35\% \\
16-8-1-1-0 & 0.468 & 0.587 & +25.58\% & 0.128 & -72.68\% \\
4-2-16-1-1 & 0.472 & 0.562 & +18.95\% & 0.134 & -71.63\% \\
4-4-8-1-0 & 0.478 & 0.524 & +9.60\% & 0.122 & -74.56\% \\
2-4-16-1-1 & 0.527 & 0.696 & +32.08\% & 0.159 & -69.90\% \\
2-4-16-1-0 & 0.538 & 0.616 & +14.49\% & 0.159 & -